Iterating through the audiogram objects

In [1]:
import csv
from io import StringIO


In [ ]:
class BaseAudiogram:
    def __init__(self, thresholds=None):
        self.thresholds = thresholds or {}

    def plot(self, side="left"):
        raise NotImplementedError("Subclasses must implement plot()")

    def to_dict(self) -> dict[int, float]:
        """Return a dictionary of frequency -> threshold."""
        return dict(self.thresholds)

    import csv
from io import StringIO
class BaseAudiogram:
    def __init__(self, thresholds=None):
        self.thresholds = thresholds or {}

    def plot(self, side="left"):
        raise NotImplementedError("Subclasses must implement plot()")

    def to_dict(self) -> dict[int, float]:
        """Return a dictionary of frequency -> threshold."""
        return dict(self.thresholds)

    @classmethod
    def from_dict(cls, data: dict[int, float]):
        """Create an Audiogram object from a dictionary."""
        return cls(thresholds=data)

    def to_json(self) -> str:
        """Serialize the audiogram to a JSON string."""
        import json
        return json.dumps(self.to_dict())

    @classmethod
    def from_json(cls, json_str: str):
        """Create an Audiogram from a JSON string."""
        import json
        data = json.loads(json_str)
        return cls.from_dict(data)

def make_threshold_property(freq):
    def getter(self):
        return self.thresholds.get(freq)

    def setter(self, value):
        self.thresholds[freq] = value

    return property(getter, setter)

# Class generator to allow iteration through threshold property creation
def make_audiogram_class(default_freqs=None):
    default_freqs = default_freqs or [250, 500, 1000, 2000, 4000, 8000]

    class Audiogram(BaseAudiogram):
        STANDARD_FREQS = default_freqs

        def __init__(self, thresholds=None, frequencies=None):
            super().__init__(thresholds)
            self._frequencies = frequencies or self.STANDARD_FREQS

            for freq in self._frequencies:
                attr = f"thres{freq}"
                if not hasattr(self.__class__, attr):
                    setattr(self.__class__, attr, make_threshold_property(freq))

        def plot(self, side="left"):
            import matplotlib.pyplot as plt

            if not self.thresholds:
                raise ValueError("No thresholds available to plot.")

            freqs = sorted(self.thresholds.keys())
            values = [self.thresholds[f] for f in freqs]

            marker = 'x' if side == "left" else 'o'
            color = 'blue' if side == "left" else 'red'

            plt.figure(figsize=(8, 6))
            plt.plot(freqs, values, marker=marker, color=color, linestyle='-')
            plt.gca().invert_yaxis()
            plt.xticks(freqs)
            plt.yticks(range(-10, 120, 10))
            plt.xlabel("Frequency (Hz)")
            plt.ylabel("Hearing Threshold (dB HL)")
            plt.title(f"{side.capitalize()} Ear Audiogram")
            plt.grid(True)
            plt.show()

    for freq in default_freqs:
        setattr(Audiogram, f"thres{freq}", make_threshold_property(freq))

    return Audiogram


# This is what users will import:
Audiogram = make_audiogram_class()


class BinauralAudiogram:
    def __init__(self, left: BaseAudiogram, right: BaseAudiogram):
        self.left = left
        self.right = right
        self.L = self.left
        self.R = self.right

    def get_threshold(self, freq: int, side: str):
        ear = {"left": self.left, "right": self.right}.get(side.lower())
        if not ear:
            raise ValueError("Side must be 'left' or 'right'")
        return ear.thresholds.get(freq)

    def symmetry(self) -> dict[int, float]:
        """Return frequency-to-difference map (left - right)"""
        common_freqs = set(self.left.thresholds) & set(self.right.thresholds)
        return {
            freq: self.left.thresholds[freq] - self.right.thresholds[freq]
            for freq in common_freqs
        }

    def to_dict(self) -> dict[str, dict[int, float]]:
        """Return a dictionary with 'left' and 'right' keys mapping to each ear's data."""
        return {
            "left": self.left.to_dict(),
            "right": self.right.to_dict()
        }

    @classmethod
    def from_dict(cls, data: dict[int, float]):
        """Create an Audiogram object from a dictionary."""
        # Coerce keys to int so JSON and CSV inputs work cleanly
        cleaned = {int(k): v for k, v in data.items()}
        return cls(thresholds=cleaned)

    def to_json(self) -> str:
        """Serialize the binaural audiogram to a JSON string."""
        import json
        return json.dumps(self.to_dict())

    @classmethod
    def from_json(cls, json_str: str):
        """Construct a BinauralAudiogram from a JSON string."""
        import json
        data = json.loads(json_str)
        return cls.from_dict(data)


    @classmethod
    def from_wide_dict(cls, data: dict[str, float]):
        """Parse wide-format dict with keys like 'L_500', 'R_500' into a BinauralAudiogram."""
        left_data = {}
        right_data = {}

        for key, value in data.items():
            if key.startswith("L_"):
                try:
                    freq = int(key[2:])
                    left_data[freq] = value
                except ValueError:
                    continue
            elif key.startswith("R_"):
                try:
                    freq = int(key[2:])
                    right_data[freq] = value
                except ValueError:
                    continue

        left = Audiogram.from_dict(left_data)
        right = Audiogram.from_dict(right_data)
        return cls(left, right)

    @classmethod
    def from_csv(cls, csv_str: str):
        """
        Create a BinauralAudiogram from a CSV string.
        Assumes a single row with columns like L_250, R_250, etc.
        """
        reader = csv.DictReader(StringIO(csv_str))
        rows = list(reader)
        if not rows:
            raise ValueError("CSV must contain at least one row.")
        return cls.from_wide_dict(rows[0])

    def to_csv(self) -> str:
        """
        Export the BinauralAudiogram as a one-row CSV string.
        """
        wide_dict = {}
        for freq, val in self.left.to_dict().items():
            wide_dict[f"L_{freq}"] = val
        for freq, val in self.right.to_dict().items():
            wide_dict[f"R_{freq}"] = val

        output = StringIO()
        writer = csv.DictWriter(output, fieldnames=sorted(wide_dict.keys()))
        writer.writeheader()
        writer.writerow(wide_dict)
        return output.getvalue()

    def to_json(self) -> str:
        """Serialize the audiogram to a JSON string."""
        import json
        return json.dumps(self.to_dict())

    @classmethod
    def from_json(cls, json_str: str):
        """Create an Audiogram from a JSON string."""
        import json
        data = json.loads(json_str)
        return cls.from_dict(data)

def make_threshold_property(freq):
    def getter(self):
        return self.thresholds.get(freq)

    def setter(self, value):
        self.thresholds[freq] = value

    return property(getter, setter)

In [3]:
# Class generator to allow iteration through threshold property creation
def make_audiogram_class(default_freqs=None):
    default_freqs = default_freqs or [250, 500, 1000, 2000, 4000, 8000]

    class Audiogram(BaseAudiogram):
        STANDARD_FREQS = default_freqs

        def __init__(self, thresholds=None, frequencies=None):
            super().__init__(thresholds)
            self._frequencies = frequencies or self.STANDARD_FREQS

            for freq in self._frequencies:
                attr = f"thres{freq}"
                if not hasattr(self.__class__, attr):
                    setattr(self.__class__, attr, make_threshold_property(freq))

        def plot(self, side="left"):
            import matplotlib.pyplot as plt

            if not self.thresholds:
                raise ValueError("No thresholds available to plot.")

            freqs = sorted(self.thresholds.keys())
            values = [self.thresholds[f] for f in freqs]

            marker = 'x' if side == "left" else 'o'
            color = 'blue' if side == "left" else 'red'

            plt.figure(figsize=(8, 6))
            plt.plot(freqs, values, marker=marker, color=color, linestyle='-')
            plt.gca().invert_yaxis()
            plt.xticks(freqs)
            plt.yticks(range(-10, 120, 10))
            plt.xlabel("Frequency (Hz)")
            plt.ylabel("Hearing Threshold (dB HL)")
            plt.title(f"{side.capitalize()} Ear Audiogram")
            plt.grid(True)
            plt.show()

    for freq in default_freqs:
        setattr(Audiogram, f"thres{freq}", make_threshold_property(freq))

    return Audiogram


# This is what users will import:
Audiogram = make_audiogram_class()

In [4]:
class BinauralAudiogram:
    def __init__(self, left: BaseAudiogram, right: BaseAudiogram):
        self.left = left
        self.right = right
        self.L = self.left
        self.R = self.right

    def get_threshold(self, freq: int, side: str):
        ear = {"left": self.left, "right": self.right}.get(side.lower())
        if not ear:
            raise ValueError("Side must be 'left' or 'right'")
        return ear.thresholds.get(freq)

    def symmetry(self) -> dict[int, float]:
        """Return frequency-to-difference map (left - right)"""
        common_freqs = set(self.left.thresholds) & set(self.right.thresholds)
        return {
            freq: self.left.thresholds[freq] - self.right.thresholds[freq]
            for freq in common_freqs
        }

    def to_dict(self) -> dict[str, dict[int, float]]:
        """Return a dictionary with 'left' and 'right' keys mapping to each ear's data."""
        return {
            "left": self.left.to_dict(),
            "right": self.right.to_dict()
        }

    @classmethod
    def from_dict(cls, data: dict[str, dict[int, float]]):
        """Construct a BinauralAudiogram from a nested dict like {'left': {...}, 'right': {...}}"""
        left = Audiogram.from_dict(data.get("left", {}))
        right = Audiogram.from_dict(data.get("right", {}))
        return cls(left, right)

    def to_json(self) -> str:
        """Serialize the binaural audiogram to a JSON string."""
        import json
        return json.dumps(self.to_dict())

    @classmethod
    def from_json(cls, json_str: str):
        """Construct a BinauralAudiogram from a JSON string."""
        import json
        data = json.loads(json_str)
        return cls.from_dict(data)


    @classmethod
    def from_wide_dict(cls, data: dict[str, float]):
        """Parse wide-format dict with keys like 'L_500', 'R_500' into a BinauralAudiogram."""
        left_data = {}
        right_data = {}

        for key, value in data.items():
            if key.startswith("L_"):
                try:
                    freq = int(key[2:])
                    left_data[freq] = value
                except ValueError:
                    continue
            elif key.startswith("R_"):
                try:
                    freq = int(key[2:])
                    right_data[freq] = value
                except ValueError:
                    continue

        left = Audiogram.from_dict(left_data)
        right = Audiogram.from_dict(right_data)
        return cls(left, right)

    @classmethod
    def from_csv(cls, csv_str: str):
        """
        Create a BinauralAudiogram from a CSV string.
        Assumes a single row with columns like L_250, R_250, etc.
        """
        reader = csv.DictReader(StringIO(csv_str))
        rows = list(reader)
        if not rows:
            raise ValueError("CSV must contain at least one row.")
        return cls.from_wide_dict(rows[0])

    def to_csv(self) -> str:
        """
        Export the BinauralAudiogram as a one-row CSV string.
        """
        wide_dict = {}
        for freq, val in self.left.to_dict().items():
            wide_dict[f"L_{freq}"] = val
        for freq, val in self.right.to_dict().items():
            wide_dict[f"R_{freq}"] = val

        output = StringIO()
        writer = csv.DictWriter(output, fieldnames=sorted(wide_dict.keys()))
        writer.writeheader()
        writer.writerow(wide_dict)
        return output.getvalue()